<a href="https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

To group the web pages into different archetypes (like Protect or Rewrite), I need features that describe the content. I am choosing word_count, search_volume, competition, and cpc (cost per click). I will fill any missing empty values with 0. Then, I will scale the numbers so large values do not break the clustering math.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from sklearn.preprocessing import StandardScaler
from google.colab import userdata

# 1. Connect to Hugging Face
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")

# 2. Get the data (pulling a sample of 5000 rows for building)
query = """
    SELECT word_count, search_volume, competition, cpc, client_hash_id, url_hash_id
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    LIMIT 5000
"""
df_raw = con.execute(query).df()

# 3. Build the feature vector
feature_cols = ['word_count', 'search_volume', 'competition', 'cpc']
df_features = df_raw[feature_cols].copy()

# Fill missing values with 0
df_features = df_features.fillna(0)

# Scale the data for clustering
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=feature_cols)

print("Feature vector built. Data shape:", df_scaled.shape)
df_scaled.head()

Feature vector built. Data shape: (5000, 4)


,word_count,search_volume,competition,cpc
0,-0.251888,-0.145386,1.908621,0.039664
1,-0.823439,-0.168347,-0.588734,-0.118035
2,0.159628,0.371242,0.399231,-0.018266
3,-0.402778,-0.179828,-0.588734,-0.118035
4,-0.265605,2.575522,1.332309,0.026790


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

word_count: The total number of words on the web page. Missing values are filled with 0. It is available before the model runs.

search_volume: How many times people search for this topic. Missing values are filled with 0. It is available before the model runs.

competition: A score of how hard it is to rank for this topic. Missing values are filled with 0. It is available before the model runs.

cpc: The cost per click for the topic. Missing values are filled with 0. It is available before the model runs.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check that there are no missing values left in our features
missing_counts = df_features.isna().sum()
print("Total missing values in each feature column:")
print(missing_counts)

Total missing values in each feature column:
word_count       0
search_volume    0
competition      0
cpc              0
dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage happens if the model accidentally uses private information or future answers. For clustering, the biggest risk is grouping pages just by who owns them. I will write a test to prove that the private client IDs and URLs are completely removed from my scaled feature vector.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# The Leakage Test
private_columns = ['client_hash_id', 'url_hash_id', 'content_hash_id']
leaks_found = [col for col in private_columns if col in df_scaled.columns]

if len(leaks_found) == 0:
    print("Test Passed: No private IDs or text strings found in the feature vector.")
else:
    print("WARNING: Leakage found! Please remove these columns:", leaks_found)


Test Passed: No private IDs or text strings found in the feature vector.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

client_hash_id: Excluded to follow the public safety rules. The model must look at content performance, not the company name.

url_hash_id: Excluded because random text strings do not work in mathematical clustering models.

content_created_date: Excluded because I am grouping content by its value and size, not by the calendar date it was made.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Final confirmation of the clean columns sent to the model
print("Final columns ready for clustering:")
print(df_scaled.columns.tolist())


Final columns ready for clustering:
['word_count', 'search_volume', 'competition', 'cpc']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.